# AIRLINE PREDICTION

Aquí mostraré el código utilizado para realizar las predicciones

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


DATA_DIR = Path(__file__).resolve().parent
train_data = pd.read_csv(DATA_DIR / "train.csv")
test_data = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if not sample_submission["id"].equals(test_data["id"]):
    raise ValueError("Los IDs de la plantilla no coinciden con test.csv.")

y = train_data["satisfaction"].astype("int8")
categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class"]
delay_col = "Arrival Delay in Minutes"
rating_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
]


def add_features(frame):
    frame = frame.drop(columns=["id"], errors="ignore").copy()
    frame["ArrivalDelayMissing"] = frame[delay_col].isna().astype("int8")
    frame["ServiceRatingMean"] = frame[rating_cols].mean(axis=1)
    frame["ServiceRatingZeroCount"] = (frame[rating_cols] == 0).sum(axis=1)
    total_delay = frame["Departure Delay in Minutes"].fillna(0) + frame[delay_col].fillna(0)
    frame["TotalDelayLog1p"] = np.log1p(total_delay)
    frame["TravelClass"] = frame["Type of Travel"] + "_" + frame["Class"]
    return frame

X_raw = add_features(train_data.drop(columns=["satisfaction"]))
X_test_raw = add_features(test_data)
X_lgb = pd.get_dummies(
    X_raw, columns=categorical_cols + ["TravelClass"], dtype="uint8"
)
X_test_lgb = pd.get_dummies(
    X_test_raw, columns=categorical_cols + ["TravelClass"], dtype="uint8"
).reindex(columns=X_lgb.columns, fill_value=0)


def make_lightgbm(random_state, n_estimators=3000):
    return LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        learning_rate=0.025,
        num_leaves=63,
        min_child_samples=30,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=random_state,
        verbosity=-1,
    )


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(train_data), dtype="float32")
lgb_best_iterations = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X_raw, y), start=1):
    y_fold_train = y.iloc[train_idx]
    y_fold_valid = y.iloc[valid_idx]

    X_fold_train = X_lgb.iloc[train_idx].copy()
    X_fold_valid = X_lgb.iloc[valid_idx].copy()
    fold_median = X_fold_train[delay_col].median()
    X_fold_train[delay_col] = X_fold_train[delay_col].fillna(fold_median)
    X_fold_valid[delay_col] = X_fold_valid[delay_col].fillna(fold_median)

    lgb_model = make_lightgbm(random_state=42 + fold)
    print(f"Entrenando LightGBM, fold {fold}/5...", flush=True)
    lgb_model.fit(
        X_fold_train,
        y_fold_train,
        eval_X=X_fold_valid,
        eval_y=y_fold_valid,
        eval_metric="auc",
        callbacks=[early_stopping(150, verbose=False)],
    )
    oof_lgb[valid_idx] = lgb_model.predict_proba(
        X_fold_valid, num_iteration=lgb_model.best_iteration_
    )[:, 1]
    lgb_best_iterations.append(lgb_model.best_iteration_)

    fold_auc = roc_auc_score(y_fold_valid, oof_lgb[valid_idx])
    print(
        f"Fold {fold} ROC AUC: {fold_auc:.6f} "
        f"({lgb_model.best_iteration_} árboles)",
        flush=True,
    )

auc_lgb = roc_auc_score(y, oof_lgb)
print(f"\nLightGBM OOF ROC AUC: {auc_lgb:.6f}")

final_median = X_lgb[delay_col].median()
X_lgb_full = X_lgb.copy()
X_lgb_test = X_test_lgb.copy()
X_lgb_full[delay_col] = X_lgb_full[delay_col].fillna(final_median)
X_lgb_test[delay_col] = X_lgb_test[delay_col].fillna(final_median)

final_model = make_lightgbm(
    random_state=42,
    n_estimators=max(100, int(np.median(lgb_best_iterations))),
)
final_model.fit(X_lgb_full, y)
probabilities = final_model.predict_proba(X_lgb_test)[:, 1]
print(f"Modelo final: LightGBM ({int(np.median(lgb_best_iterations))} árboles)")

submission = pd.DataFrame({
    "id": test_data["id"],
    "satisfaction": probabilities,
})
submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission guardada en: {submission_path}")

: 

# Resultados

Y aquí muestro los resultados obtenidos

![Resultados](img/image.png)

Aqui la clasificaión grupal

![Grupal](img/grupal.png)